[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reinhart-group/generative-copolymer-workshop/blob/main/day2/04_embedding.ipynb)

# Day 2 — Morning: PCA, UMAP & Latent Space

**Objectives:**
- Understand the curse of dimensionality in soft matter
- Compare linear (PCA) vs. non-linear (UMAP) dimensionality reduction
- Visualize the continuous latent space ($Z$) of morphologies
- Identify structural regions: strings, vesicles, spherical micelles



**The big picture:** we start with a big pile of numbers describing polymer structures
(thousands of them), squeeze each structure down to a single point on a 2-D map, and then
read that map to understand which structures are similar to which. That map is what we
mean by an *embedding* or *latent space*.


## Setup

If you're on Google Colab, run the install cell first. Locally, you can skip it if the packages are already installed.


In [ ]:
# If running on Colab, uncomment and run this cell first:
# !pip install umap-learn scikit-learn tqdm

**The toolkit.** We will be using: 

- `numpy`
- `pandas`
- `matplotlib`
- `scikit-learn` (`sklearn`)
- `umap-learn` (`umap`)
- `PIL` (`Image`, `ImageOps`)




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os, ast, glob
from PIL import Image, ImageOps
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import umap

## Helper functions

The next two cells define utility functions used in the rest of the notebook. 

The first group handles feature pooling, which takes the the features for 
each view of a certain aggregate structure and averages it into one representative list. 


In [ ]:
# --- Helper functions ---

# --- Feature pooling (Section 1) ---

def beads_match(stored, target):
    """Return True if a stored bead combo (string or list) matches target, in any order.
    Parameters
    ----------
    stored : list or string
    target : list or string

    Returns
    -------
    sorted list or string

    """
    try:
        stored_list = ast.literal_eval(stored) if isinstance(stored, str) else stored
        return sorted(stored_list) == sorted(target)
    except Exception:
        return False


def pool_features(features, pool_type='mean'):
    """
    Combine a list of feature vectors into one by pooling.

    Parameters
    ----------
    features  : list of 1-D arrays
    pool_type : 'mean' or 'max'

    Returns
    -------
    1-D numpy array
    """
    features = np.array(features)
    if pool_type == 'mean':
        return np.mean(features, axis=0)
    elif pool_type == 'max':
        return np.max(features, axis=0)
    else:
        raise ValueError(f"pool_type must be 'mean' or 'max', got '{pool_type}'.")


def get_features_all_combos(df, bead_combinations, views, pool_type='mean'):
    """
    Pool CNN features across all bead-pair combinations and viewing angles.

    During featurization, each polymer structure was imaged from three orthogonal
    views (xy, xz, yz) and for multiple pairs of bead types (e.g. SN3r+C4).
    This function collects all those feature vectors for each sequence and
    averages them into a single representative vector.

    Parameters
    ----------
    df                : pd.DataFrame with columns Sequence, Beads, View, Feature
    bead_combinations : list of lists, e.g. [['SN3r','C4'], ['SN3r','TP1']]
    views             : list of str, e.g. ['xy', 'xz', 'yz']
    pool_type         : 'mean' or 'max'

    Returns
    -------
    pd.DataFrame with columns Sequence, Feature — one row per sequence.
    """
    bead_mask = df['Beads'].apply(
        lambda b: any(beads_match(b, combo) for combo in bead_combinations)
    )
    view_mask = df['View'].isin(views)
    filtered  = df[bead_mask & view_mask].copy()

    if len(filtered) == 0:
        raise ValueError('No matching rows — check bead names and view labels.')

    n_combos = filtered['Beads'].nunique()
    n_views  = filtered['View'].nunique()
    print(f'  Found {n_combos} bead combos and {n_views} views '
          f'across {filtered["Sequence"].nunique()} sequences')
    print(f'  Pooling {len(filtered)} total rows → one vector per sequence')

    records = []
    for seq, group in filtered.groupby('Sequence'):
        pooled_feat = pool_features(list(group['Feature']), pool_type)
        records.append({'Sequence': seq, 'Feature': pooled_feat})

    return pd.DataFrame(records)



The second group handles manifold visualization. These functions help us find images and represent the two-dimensional embedding we will calculate.


In [ ]:
# --- Manifold visualization (Section 4) ---

def find_image(folder_path, bead, view, name_extension):
    """Find the KDE density image for a given bead type and viewing angle."""
    for fname in os.listdir(folder_path):
        if fname.startswith(f'{bead}_{view}_') and fname.endswith(name_extension):
            return os.path.join(folder_path, fname)
    return None


def load_rgb_composite(seq_path, ch_r, ch_g, view, name_extension):
    """
    Load two single-bead KDE images and combine into an RGB image.

    R = bead type ch_r density
    G = bead type ch_g density
    B = combined density (R + G, clipped to 255)

    Returns a (H, W, 3) uint8 array, or None if either image is missing.
    """
    path_r = find_image(seq_path, ch_r, view, name_extension)
    path_g = find_image(seq_path, ch_g, view, name_extension)
    if path_r is None or path_g is None:
        return None
    r  = np.array(Image.open(path_r).convert('L'), dtype=np.uint8)
    g  = np.array(Image.open(path_g).convert('L'), dtype=np.uint8)
    b  = np.clip(r.astype(np.uint16) + g.astype(np.uint16), 0, 255).astype(np.uint8)
    return np.stack([r, g, b], axis=-1)


def find_files_glob(root_dir, seqs):
    """Return a list of matching folder paths for each sequence name."""
    found_paths = []
    for seq in seqs:
        files = glob.glob(os.path.join(root_dir, '**', seq), recursive=True)
        found_paths.append(files if files else None)
    return found_paths


def make_grid_2d_clustered(x, y, res, min_radius=None):
    """
    Select one representative point per cluster across the 2-D embedding.

    Rather than snapping to a rigid grid, this uses agglomerative clustering
    so representatives are always real data points and naturally follow the
    shape of the embedding.

    Parameters
    ----------
    x, y       : 1-D arrays of embedding coordinates
    res        : cluster distance threshold — smaller gives more points
    min_radius : minimum separation between selected representatives (default = res)

    Returns
    -------
    List of integer indices into x/y.
    """
    if min_radius is None:
        min_radius = res
    xy = np.vstack([x, y]).T
    clustering = AgglomerativeClustering(
        n_clusters=None, distance_threshold=res, linkage='complete'
    )
    labels = clustering.fit_predict(xy)

    representatives = []
    for label in np.unique(labels):
        mask     = np.where(labels == label)[0]
        centroid = xy[mask].mean(axis=0)
        best     = np.argmin(np.linalg.norm(xy[mask] - centroid, axis=1))
        representatives.append(mask[best])

    accepted, accepted_coords = [], []
    for idx, coord in zip(representatives, xy[representatives]):
        if not accepted_coords:
            accepted.append(idx)
            accepted_coords.append(coord)
            continue
        if np.linalg.norm(np.array(accepted_coords) - coord, axis=1).min() >= min_radius:
            accepted.append(idx)
            accepted_coords.append(coord)

    return accepted

## 1. Loading Feature Vectors

In Day 1, a pre-trained CNN (EfficientNet) extracted feature vectors from simulated
polymer structure images. Each sequence was imaged from **three viewing angles** (xy, xz, yz)
and for **multiple bead-type combinations** — giving 9+ feature vectors per sequence.

Here we load those per-sequence CSV files, then **pool** across all views and bead
pairs to get a single feature vector per sequence. This vector is our high-dimensional
representation of each polymer morphology.

**What's a "feature vector"?** Imagine describing a structure with a long list of numbers —
say 1280 of them — where each number measures some visual trait the neural network noticed.
That list is a *feature vector*: a numerical fingerprint of the structure. Structures that
look alike have similar fingerprints. "High-dimensional" just means the list is long (1280
numbers = 1280 dimensions), which is impossible to picture directly — hence the whole point
of this notebook, which is to shrink it down to 2 dimensions we *can* picture.

We'll load these fingerprints in four small steps.


**Step 1a — Set your options.** These variables describe how the Day-1 data was produced (which model, image size, which atom pairs and camera angles). They control which files get loaded below.


In [ ]:
# --- OPTIONS (change these to match your Day 1 setup) ---
MODEL_NAME = 'efficientnet_b0'   # model used during featurization
KDE_SIGMA  = 2.0                  # KDE smoothing used during image rendering

MODEL_IMG_SIZE = {
    'resnet50':        224,
    'efficientnet_b0': 224,
    'efficientnet_b1': 240,
    'efficientnet_b2': 260,
}
IMG_SIZE       = MODEL_IMG_SIZE[MODEL_NAME]
pool_type      = 'mean'           # how to combine features across views/beads
name_extension = f'_cv2_kde_sigma_{KDE_SIGMA}_size_{IMG_SIZE}.png'
df_file        = f'img_feats_{MODEL_NAME}{name_extension[:-4]}.csv'

# Bead types and viewing angles used during Day 1 featurization
bead_combinations = [['SN3r', 'C4'], ['SN3r', 'TP1'], ['C4', 'TP1']]
views             = ['xy', 'xz', 'yz']



**Step 1b — Find your data.** Point `your_folder_path` at the folder of per-structure results, then list the sequence sub-folders inside it.


In [ ]:
# --- PATH TO YOUR DATA ---
your_folder_path = '/noether/s1/kac6810/a_compare_embeddings/high_res_imgs/traj_data'
# (On Colab, mount Google Drive first and set this to your folder path)

# Discover sequence folders
seqs = [s for s in os.listdir(your_folder_path)
        if os.path.isdir(os.path.join(your_folder_path, s))]
print(f'Found {len(seqs)} sequences')



**Step 1c — Load the feature files.** Each sequence folder has a CSV of fingerprints. We read them all and stack them into one big table (`combined_df`). The `Feature` column arrives as text, so we parse it back into real number arrays.


In [ ]:
# Load per-sequence feature CSVs
csv_files = [os.path.join(your_folder_path, seq, df_file) for seq in seqs]
existing  = [f for f in csv_files if os.path.exists(f)]
missing   = len(csv_files) - len(existing)
print(f'Loading {len(existing)}/{len(csv_files)} feature CSVs'
      + (f'  ({missing} missing)' if missing else ''))

dfs = []
for csv_path in existing:
    df = pd.read_csv(csv_path, dtype={'Sequence': str})
    # Feature column is stored as a string list — parse it back to a numpy array
    df['Feature'] = df['Feature'].apply(
        lambda x: np.array(ast.literal_eval(x), dtype=np.float32)
    )
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)
print(f'Total rows: {len(combined_df)}  '
      f'(each row = one bead combo × one view × one sequence)')



**Step 1d — Pool into one vector per sequence.** Finally we average all the images for each sequence down to a single fingerprint, giving a clean matrix of `sequences × features`.


In [ ]:
# Pool across all bead combos and views → one feature vector per sequence
pooled_df = get_features_all_combos(combined_df, bead_combinations, views, pool_type)
features  = np.vstack(pooled_df['Feature'].values)
sequences = pooled_df['Sequence'].values

print(f'\nFeature matrix: {features.shape[0]} sequences × {features.shape[1]} features')

▶ **What you should see:** a printout ending in something like `Feature matrix: N sequences × 1280 features`. That matrix is the input to everything that follows. If `N` is 0, the path in Step 1b is probably wrong.


## 2. PCA — Linear Embedding

**Principal Component Analysis (PCA)** finds the directions of greatest variance in the
high-dimensional feature space and projects the data onto those directions.

- The first principal component (PC1) captures the most variance, PC2 the second-most, etc.
- The projection is *linear* — straight-line distances in the original space are preserved.
- PCA is fast and deterministic, but it may miss curved or folded structure in the data.

We'll look at two plots:
1. **Scree plot** — how much variance each component explains (tells us how many dimensions we really need)
2. **2-D scatter** — the polymer landscape projected onto the first two PCs

**PCA in one breath.** Our fingerprints live in 1280 dimensions, but most of those
dimensions barely change from structure to structure. PCA finds the few directions along
which the data *actually* spreads out the most, and lets us keep just those. "Variance
explained" is its way of saying *how much of the real variation each direction captures* —
a scree plot is just a bar chart of that, biggest first. If the first couple of bars are
tall, two dimensions are enough to summarize the data well.


**Step 2a — Fit PCA and read the scree plot.** The left panel shows how much each component captures; the right panel adds them up. Where the cumulative line crosses ~90% tells you how many dimensions you'd really need.


In [ ]:
# --- FIT PCA ---
pca        = PCA()
pca_coords = pca.fit_transform(features)

# ── Scree plot ────────────────────────────────────────────────────────────────
n_show = 30
cumvar = np.cumsum(pca.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, n_show + 1),
            pca.explained_variance_ratio_[:n_show] * 100,
            color='steelblue')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('Scree Plot')

axes[1].plot(range(1, n_show + 1), cumvar[:n_show] * 100,
             marker='o', color='steelblue')
axes[1].axhline(90, color='grey', linestyle='--', label='90% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_title('Cumulative Variance Explained')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'PC1 + PC2 explain {cumvar[1]*100:.1f}% of variance')
print(f'First 10 PCs explain {cumvar[9]*100:.1f}% of variance')



**Step 2b — The 2-D map.** Now we plot every structure using just the first two components. Each dot is one polymer sequence. We also save these coordinates (`pca_df`) for Section 4.


In [ ]:
# ── 2-D scatter ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(pca_coords[:, 0], pca_coords[:, 1], s=8, alpha=0.4, color='steelblue')
ax.set_xlabel(f'PC1  ({pca.explained_variance_ratio_[0]*100:.1f}% var.)')
ax.set_ylabel(f'PC2  ({pca.explained_variance_ratio_[1]*100:.1f}% var.)')
ax.set_title('PCA — 2D Embedding of Polymer Structures')
plt.tight_layout()
plt.show()

# Save for later sections
pca_df = pd.DataFrame({
    'Sequence': sequences,
    'Z0': pca_coords[:, 0],
    'Z1': pca_coords[:, 1],
})

▶ **What you should see:** two figures — the scree plot, then a scatter of dots. Don't worry if the scatter looks like a single blob; PCA is linear and often can't separate curved structure. That's exactly the limitation UMAP addresses next.


## 3. UMAP — Non-linear Embedding

**UMAP (Uniform Manifold Approximation and Projection)** is a non-linear method that tries
to preserve the *local neighborhood structure* of the data. While PCA finds globally optimal
linear directions, UMAP can "unfold" curved or branching manifolds that PCA would project
on top of each other.

Key hyperparameters:
- **`n_neighbors`** — how many nearby points each point considers when learning structure.
  Small values emphasize fine local clusters; large values give a more global view.
- **`min_dist`** — minimum distance between points in the 2-D layout.
  Small values pack points tightly; larger values spread them out.

> **Try it:** after running the default, change `n_neighbors` to 5 or 50 and re-run to see how the layout shifts.

**The intuition for UMAP:** instead of finding global straight-line directions like PCA, it
looks at each structure's nearest neighbors and tries to keep neighbors close together in the
2-D picture, even if that means bending and stretching the layout. The result is often a map
with clearer clusters. The next cell fits UMAP and draws it **side by side** with PCA so you
can compare the same data under both methods.


In [ ]:
# --- FIT UMAP ---
n_neighbors = 15    # try: 5 (local) or 50 (global)
min_dist    = 0.1   # try: 0.0 (tight) or 0.5 (spread)

reducer     = umap.UMAP(n_components=2, n_neighbors=n_neighbors,
                        min_dist=min_dist, random_state=42)
umap_coords = reducer.fit_transform(features)

# ── Side-by-side comparison ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(pca_coords[:, 0], pca_coords[:, 1],
                s=8, alpha=0.4, color='steelblue')
axes[0].set_xlabel(f'PC1  ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2  ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('PCA (linear)')

axes[1].scatter(umap_coords[:, 0], umap_coords[:, 1],
                s=8, alpha=0.4, color='coral')
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')
axes[1].set_title(f'UMAP  (n_neighbors={n_neighbors}, min_dist={min_dist})')

plt.suptitle('Linear vs Non-linear Dimensionality Reduction', fontsize=13)
plt.tight_layout()
plt.show()

# Save for Section 4
umap_df = pd.DataFrame({
    'Sequence': sequences,
    'Z0': umap_coords[:, 0],
    'Z1': umap_coords[:, 1],
})

▶ **What to look for:** the PCA panel (blue) vs the UMAP panel (coral) of the *same* structures. UMAP usually pulls apart groups that PCA piles on top of each other. 🔬 **Try it:** as the cell suggests, set `n_neighbors` to 5 or 50 and re-run to watch the layout tighten or loosen.


## 4. Interpreting the Manifold

Now that we have a 2-D embedding, what does it actually *mean*?

The best way to interpret an embedding of polymer structures is to place real simulation
snapshots onto the map. Points that look similar in the embedding should correspond to
sequences with similar morphologies (strings, vesicles, spherical micelles, etc.).

The cell below samples a spatially even grid of representative points from the embedding,
finds the corresponding simulation snapshot for each, and places those thumbnails on the plot.

> **Note:** this requires the high-resolution snapshot images from your data folder.
> If you don't have them yet, just look at the scatter plot and try to infer structure
> from the clusters.

**Why this step matters.** So far the maps are just dots — we've trusted that "close = similar"
without checking. Here we *verify* it by pasting a real simulation snapshot onto each region of
the map. If the map is meaningful, neighboring thumbnails should show similar shapes (e.g. a
patch of vesicles here, a patch of string-like structures there). The word **manifold** is just
the technical name for this curved 2-D sheet the structures live on.

We do it in two steps: first set up which map and which images to use, then loop over a
spread-out sample of points and drop a thumbnail on each.


**Step 4a — Choose the map and image settings.** Pick `pca_df` or `umap_df`, set the snapshot filename pattern, and sample a spread-out set of representative points (so thumbnails don't overlap). `res` controls how many you get.


In [ ]:
# --- CHOOSE EMBEDDING ---
# Switch between pca_df and umap_df to compare how the two layouts look
# with actual morphology images placed on them.
embed_df = pca_df.copy()   # or: umap_df.copy()

x, y = embed_df[['Z0', 'Z1']].values.T

# --- SNAPSHOT IMAGE PARAMETERS ---
# Adjust these to match the filenames of the images rendered in Day 1
s              = 36
fig_size       = (10, 10)
dpi            = 500
snapshot_extension = f'_high_res_size_{s}_figsize_{fig_size[0]}_{fig_size[1]}_dpi_{dpi}.png'
snapshot_view  = 'px'        # which viewing angle to display
snapshot_beads = 'C4_SN3r'  # which bead-pair color channel to display

# --- GRID SAMPLING ---
res  = 0.03    # grid spacing: decrease for more images, increase for fewer
zoom = 0.008   # how large each thumbnail appears on the plot
pad  = 0.01    # padding around each thumbnail box

best_idx       = make_grid_2d_clustered(x, y, res)
best_idx_df    = embed_df.iloc[best_idx].reset_index()
seq_at_indices = embed_df.iloc[best_idx]['Sequence']
found_path_dir = find_files_glob(your_folder_path, seq_at_indices)

print(f'Placing {len(best_idx)} snapshots on the manifold...')



**Step 4b — Drop snapshots onto the map.** This loops over the chosen points, finds each one's image file, and places it as a small thumbnail at its coordinates. Missing images are counted and skipped rather than crashing the plot.


In [ ]:
# --- PLOT ---
box_props = dict(edgecolor='lightgrey', facecolor='none', linewidth=1)
fig, ax   = plt.subplots(figsize=(10, 7))

n_placed, n_skipped = 0, 0
for idx, row in best_idx_df.iterrows():
    px  = row['Z0']
    py  = row['Z1']
    seq = row['Sequence']
    png = f'{snapshot_beads}_{snapshot_view}_{seq}{snapshot_extension}'

    try:
        png_folder = found_path_dir[idx][0]
    except (TypeError, IndexError):
        n_skipped += 1
        continue

    png_path = os.path.join(png_folder, png)
    try:
        img      = Image.open(png_path)
        img      = ImageOps.crop(img, border=50)
        imagebox = OffsetImage(np.array(img), zoom=zoom)
        ab = AnnotationBbox(imagebox, (px, py),
                            frameon=True, pad=pad, bboxprops=box_props)
        ax.add_artist(ab)
        n_placed += 1
    except FileNotFoundError:
        n_skipped += 1

ax.set_xlim(x.min() - 0.05, x.max() + 0.05)
ax.set_ylim(y.min() - 0.05, y.max() + 0.025)
ax.set_xlabel('Z0')
ax.set_ylabel('Z1')
ax.set_title('Polymer Morphology Manifold')
plt.tight_layout()
plt.show()

print(f'Placed {n_placed} images, skipped {n_skipped} (missing files).')

▶ **What you should see:** the map with little structure images scattered across it, plus a line like `Placed X images, skipped Y`. A high skip count just means those image files weren't found — the plot still works for whatever it did find.


## 5. Identify Structural Regions



The manifold in Section 4 gives a bird's-eye view of the whole dataset — every structure is a dot. Here we zoom in: we sample representative sequences from different parts of the embedding and display their simulation snapshots in a grid whose layout matches the map. This turns the abstract scatter into a labelled picture atlas, and lets us answer the question *what structure does each region actually correspond to?*

**Three morphology types to watch for:**
- **Spherical micelles** — compact round blobs (dense hydrophobic core, thin corona)
- **Worm-like micelles / strings** — elongated or branching filaments
- **Liquid droplets** — blobs with no defined layering or detailed structure

**Step 5a — Build the morphology atlas.** The cell below divides the 2-D embedding into an *n* × *n* spatial grid and selects one representative sequence from each cell — the one whose coordinates are closest to the cell centre. It displays those snapshots in the same grid layout, so image position directly mirrors map position. Think of it as turning the abstract dot-cloud into a picture atlas: each region gets a face.

> **Try it:** change `n_grid` to 3 (fewer, larger tiles) or 6 (more, smaller tiles) and re-run. You can also swap `inspect_df` between `pca_df` and `umap_df` to see whether the two maps organize morphologies differently.

*Note: this step uses the `snapshot_beads`, `snapshot_view`, and `snapshot_extension` variables set in Step 4a — make sure you have run that cell first.*

In [ ]:
# --- CHOOSE EMBEDDING TO INSPECT ---
inspect_df = pca_df.copy()   # or: umap_df.copy()

x_all = inspect_df['Z0'].values
y_all = inspect_df['Z1'].values

# --- GRID PARAMETERS ---
n_grid = 4   # n×n regions — try 3 (fewer tiles) or 6 (more tiles)

x_edges = np.linspace(x_all.min(), x_all.max(), n_grid + 1)
y_edges = np.linspace(y_all.min(), y_all.max(), n_grid + 1)

# Pick the sequence nearest the centre of each grid cell.
# Rows run top→bottom (high Z1 → low Z1) to match scatter-plot orientation.
picked_indices = []
for j in range(n_grid - 1, -1, -1):       # high Z1 first (top row)
    for i in range(n_grid):               # low Z0 → high Z0 (left → right)
        in_cell = np.where(
            (x_all >= x_edges[i]) & (x_all < x_edges[i + 1]) &
            (y_all >= y_edges[j]) & (y_all < y_edges[j + 1])
        )[0]
        if len(in_cell) == 0:
            picked_indices.append(None)
            continue
        cx = (x_edges[i] + x_edges[i + 1]) / 2
        cy = (y_edges[j] + y_edges[j + 1]) / 2
        dists = (x_all[in_cell] - cx) ** 2 + (y_all[in_cell] - cy) ** 2
        picked_indices.append(in_cell[np.argmin(dists)])

# --- DISPLAY ---
fig, axes = plt.subplots(n_grid, n_grid, figsize=(14, 14))
n_shown, n_skipped = 0, 0

for cell_idx, pt_idx in enumerate(picked_indices):
    row_i, col_j = cell_idx // n_grid, cell_idx % n_grid
    ax = axes[row_i, col_j]
    ax.axis('off')

    if pt_idx is None:
        ax.set_title('(no data)', fontsize=8, color='grey')
        continue

    row    = inspect_df.iloc[pt_idx]
    seq    = row['Sequence']
    z0, z1 = row['Z0'], row['Z1']
    paths  = find_files_glob(your_folder_path, [seq])

    try:
        folder = paths[0][0]
    except (TypeError, IndexError):
        n_skipped += 1
        ax.set_title(f'Z=({z0:.2f}, {z1:.2f})\nmissing', fontsize=7, color='red')
        continue

    png_path = os.path.join(folder,
                            f'{snapshot_beads}_{snapshot_view}_{seq}{snapshot_extension}')
    try:
        ax.imshow(np.array(Image.open(png_path)))
        ax.set_title(f'Z=({z0:.2f}, {z1:.2f})', fontsize=8)
        n_shown += 1
    except FileNotFoundError:
        n_skipped += 1
        ax.set_title(f'Z=({z0:.2f}, {z1:.2f})\nmissing', fontsize=7, color='red')

fig.text(0.5,  0.00, '← low Z0                                        high Z0 →',
         ha='center', fontsize=11)
fig.text(0.00, 0.5,  '← low Z1                                        high Z1 →',
         va='center', rotation='vertical', fontsize=11)

plt.suptitle(
    f'Morphology atlas — one image per region of the {n_grid}×{n_grid} embedding grid',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.show()
print(f'Displayed {n_shown} images ({n_skipped} missing).')

▶ **What to look for.** The image grid is oriented to match the embedding: the top-left tile corresponds to the top-left corner of the scatter plot, and so on. Structures that are neighbors in the map should look similar in the grid.

**Three morphology types to find:**
- **Spherical micelles** — compact, roughly round blobs with a dense hydrophobic core and a thin corona
- **Worm-like micelles / strings** — elongated or branching filaments that meander across the frame
- **Liquid droplets** — single blobs with no discernable layers or structure

> **Discussion:** which morphology occupies the largest region of the map? Does the boundary between morphology types look sharp or gradual? Try switching `inspect_df` between `pca_df` and `umap_df` — does one embedding group morphologies more cleanly than the other?